In [1]:
"""
EDA Week 1 - Bank Customer 360 Analysis
======================================
Track: Fraud & Anomaly Detection (A) / NBFO (B) / Segmentation (C)
Team: ML End Term
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12



## 1. DATA LOADING


In [2]:
print("Loading data...")
cus = pd.read_csv("Data_Customer.csv", low_memory=False)
trans = pd.read_csv("Data_Transaction.csv", low_memory=False)
act = pd.read_csv("Data_Activity.csv", low_memory=False)
dep = pd.read_csv("Data_Deposit.csv", low_memory=False)
lend = pd.read_csv("Data_Lending.csv", low_memory=False)
card = pd.read_csv("Data_Card.csv", low_memory=False)

# Parse dates
trans['TRANS_DATE'] = pd.to_datetime(trans['TRANS_DATE'])
act['ACTIVITY_DATE'] = pd.to_datetime(act['ACTIVITY_DATE'])
dep['MONTH'] = pd.to_datetime(dep['MONTH'])
lend['MONTH'] = pd.to_datetime(lend['MONTH'])
card['MONTH'] = pd.to_datetime(card['MONTH'])

print(f"Customer: {cus.shape}")
print(f"Transaction: {trans.shape}")
print(f"Activity: {act.shape}")
print(f"Deposit: {dep.shape}")
print(f"Lending: {lend.shape}")
print(f"Card: {card.shape}")



Loading data...
Customer: (290223, 9)
Transaction: (1418030, 8)
Activity: (16132675, 6)
Deposit: (1258424, 6)
Lending: (576431, 4)
Card: (871589, 4)


## 2. DATA QUALITY - MISSING VALUES HEATMAP


In [3]:
print("\nGenerating visualizations...")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Missing values per table
tables = {
    'Customer': cus,
    'Transaction': trans,
    'Activity': act,
    'Deposit': dep,
    'Lending': lend,
    'Card': card
}

for ax, (name, df) in zip(axes.flat, tables.items()):
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100)
    missing_df = pd.DataFrame({'Count': missing, 'Pct': missing_pct})
    missing_df = missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

    if len(missing_df) > 0:
        bars = ax.barh(range(len(missing_df)), missing_df['Pct'].values,
                       color=sns.color_palette("Reds_r", len(missing_df)))
        ax.set_yticks(range(len(missing_df)))
        ax.set_yticklabels(missing_df.index, fontsize=8)
        for i, (pct, cnt) in enumerate(zip(missing_df['Pct'].values, missing_df['Count'].values)):
            ax.text(pct + 0.5, i, f'{pct:.1f}% ({cnt:,})', va='center', fontsize=7)
    else:
        ax.text(0.5, 0.5, 'No Missing Values', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(f'{name} - Missing Values (%)')
    ax.set_xlabel('% Missing')

plt.suptitle('3.1 Missing Values Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fig1_missing_values.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig1_missing_values.png")




Generating visualizations...
  -> Saved eda_fig1_missing_values.png


## 3. UNIVARIATE - CUSTOMER DEMOGRAPHICS


In [4]:
fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 3, figure=fig)

# Gender pie
ax1 = fig.add_subplot(gs[0, 0])
gender = cus['CLIENT_SEX'].value_counts()
ax1.pie(gender.values, labels=['Male', 'Female'], autopct='%1.1f%%',
        colors=['#4ECDC4', '#FF6B6B'], explode=(0.02, 0.02),
        textprops={'fontsize': 11})
ax1.set_title('Gender Distribution', fontweight='bold')

# Staff pie
ax2 = fig.add_subplot(gs[0, 1])
staff = cus['STAFF'].value_counts()
ax2.pie(staff.values, labels=['Non-Staff', 'Staff'], autopct='%1.1f%%',
        colors=['#95E1D3', '#F38181'], explode=(0.02, 0.02),
        textprops={'fontsize': 11})
ax2.set_title('Staff vs Non-Staff', fontweight='bold')

# Verify method
ax3 = fig.add_subplot(gs[0, 2])
verify = cus['VERIFY_METHOD'].value_counts()
colors_vf = ['#2EC4B6', '#E71D36', '#FF9F1C']
ax3.bar(verify.index, verify.values, color=colors_vf, edgecolor='white')
ax3.set_title('Verification Method', fontweight='bold')
ax3.set_ylabel('Count')
for i, v in enumerate(verify.values):
    ax3.text(i, v + 5000, f'{v:,}\n({v/len(cus)*100:.1f}%)', ha='center', fontsize=9)

# Register channel
ax4 = fig.add_subplot(gs[1, 0])
channel = cus['EB_REGISTER_CHANNEL'].value_counts()
colors_ch = ['#011627', '#FF3366', '#2EC4B6', '#F6F7D7']
ax4.bar(range(len(channel)), channel.values, color=colors_ch[:len(channel)], edgecolor='white')
ax4.set_xticks(range(len(channel)))
ax4.set_xticklabels(channel.index, rotation=30, ha='right', fontsize=8)
ax4.set_title('E-Bank Register Channel', fontweight='bold')
for i, v in enumerate(channel.values):
    ax4.text(i, v + 2000, f'{v:,}', ha='center', fontsize=8)

# SMS registration
ax5 = fig.add_subplot(gs[1, 1])
sms = cus['SMS'].value_counts()
ax5.pie(sms.values, labels=['Yes', 'No'], autopct='%1.1f%%',
        colors=['#2EC4B6', '#FF9F1C'], explode=(0.02, 0.02),
        textprops={'fontsize': 11})
ax5.set_title('SMS Notification Registration', fontweight='bold')

# Product depth
ax6 = fig.add_subplot(gs[1, 2])
cus_ids = set(cus['CUSTOMER_NUMBER'].unique())
trans_ids = set(trans['CUSTOMER_NUMBER'].unique())
act_ids = set(act['CUSTOMER_NUMBER'].unique())
dep_ids = set(dep['CUSTOMER_NUMBER'].unique())
lend_ids = set(lend['CUSTOMER_NUMBER'].unique())
card_ids = set(card['CUSTOMER_NUMBER'].unique())

coverage = {
    'Transaction': len(cus_ids & trans_ids),
    'Activity': len(cus_ids & act_ids),
    'Deposit': len(cus_ids & dep_ids),
    'Lending': len(cus_ids & lend_ids),
    'Card': len(cus_ids & card_ids)
}
bars = ax6.bar(coverage.keys(), coverage.values(), color=sns.color_palette("viridis", 5), edgecolor='white')
ax6.set_title('Customer Coverage by Module', fontweight='bold')
ax6.set_ylabel('Customers')
ax6.tick_params(axis='x', rotation=30)
for bar, v in zip(bars, coverage.values()):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
             f'{v:,}\n({v/len(cus_ids)*100:.1f}%)', ha='center', fontsize=8)

plt.suptitle('4.1 Customer Demographics & Module Coverage', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fig2_customer_demographics.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig2_customer_demographics.png")



  -> Saved eda_fig2_customer_demographics.png


## 4. UNIVARIATE - TRANSACTION ANALYSIS


In [5]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Transaction categories LV1
ax = axes[0, 0]
lv1 = trans['TRANS_LV1'].value_counts()
bars = ax.bar(lv1.index, lv1.values, color=['#FF6B6B', '#4ECDC4', '#FFD93D'], edgecolor='white')
ax.set_title('Transaction Categories (LV1)', fontweight='bold')
ax.set_ylabel('Count')
for bar, v in zip(bars, lv1.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'{v:,}\n({v/len(trans)*100:.1f}%)', ha='center', fontsize=9)

# Transaction LV2
ax = axes[0, 1]
lv2 = trans['TRANS_LV2'].value_counts().head(8)
bars = ax.barh(range(len(lv2)), lv2.values, color=sns.color_palette("coolwarm", len(lv2)))
ax.set_yticks(range(len(lv2)))
ax.set_yticklabels(lv2.index, fontsize=8)
ax.set_title('Top 8 Transaction Sub-Categories (LV2)', fontweight='bold')
ax.set_xlabel('Count')
for i, v in enumerate(lv2.values):
    ax.text(v + 5000, i, f'{v:,}', va='center', fontsize=8)

# Transaction amount distribution (log scale)
ax = axes[0, 2]
amounts = trans['TRANS_AMOUNT']
amounts_log = np.log10(amounts[amounts > 0])
ax.hist(amounts_log, bins=100, color='#4ECDC4', edgecolor='white', alpha=0.8)
ax.axvline(amounts_log.mean(), color='red', linestyle='--', label=f'Mean: {10**amounts_log.mean():,.0f}')
ax.axvline(amounts_log.median(), color='blue', linestyle='--', label=f'Median: {10**amounts_log.median():,.0f}')
ax.set_title('Transaction Amount Distribution (Log10)', fontweight='bold')
ax.set_xlabel('log10(Amount)')
ax.set_ylabel('Frequency')
ax.legend(fontsize=8)

# Transaction by day of week
ax = axes[1, 0]
dow_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow = trans['DAY_OF_WEEK'].value_counts().reindex(dow_order)
colors_dow = ['#FF6B6B' if d in ['Sat', 'Sun'] else '#4ECDC4' for d in dow_order]
ax.bar(dow.index, dow.values, color=colors_dow, edgecolor='white')
ax.set_title('Transactions by Day of Week', fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate(dow.values):
    ax.text(i, v + 2000, f'{v:,}\n({v/len(trans)*100:.1f}%)', ha='center', fontsize=8)

# Transaction by hour
ax = axes[1, 1]
hourly = trans.groupby('TRANS_HOUR').size()
colors_hour = ['#E71D36' if h >= 22 or h <= 5 else '#4ECDC4' for h in hourly.index]
ax.bar(hourly.index, hourly.values, color=colors_hour, edgecolor='white', width=0.8)
ax.set_title('Transactions by Hour (Red = Night 22h-5h)', fontweight='bold')
ax.set_xlabel('Hour')
ax.set_ylabel('Count')
ax.axvline(x=5.5, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=21.5, color='gray', linestyle=':', alpha=0.5)

# Transaction amount by LV1 (boxplot)
ax = axes[1, 2]
trans_sample = trans.sample(10000, random_state=42)  # Sample for performance
lv1_order = trans_sample.groupby('TRANS_LV1')['TRANS_AMOUNT'].median().sort_values(ascending=False).index
bp = ax.boxplot([trans_sample[trans_sample['TRANS_LV1']==cat]['TRANS_AMOUNT'].values
                  for cat in lv1_order],
                 labels=lv1_order, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#FF6B6B', '#4ECDC4', '#FFD93D']):
    patch.set_facecolor(color)
ax.set_title('Transaction Amount by Category (Sample 10k)', fontweight='bold')
ax.set_ylabel('Amount')
ax.set_yscale('log')
ax.tick_params(axis='x', rotation=15)

plt.suptitle('4.2 Transaction Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fig3_transaction_analysis.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig3_transaction_analysis.png")



  -> Saved eda_fig3_transaction_analysis.png


## 5. UNIVARIATE - ACTIVITY ANALYSIS


In [6]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Top activities
ax = axes[0, 0]
top_act = act['ACTIVITY_NAME'].value_counts().head(10)
bars = ax.barh(range(len(top_act)), top_act.values, color=sns.color_palette("viridis", len(top_act)))
ax.set_yticks(range(len(top_act)))
ax.set_yticklabels([x.replace('_', ' ').title()[:30] for x in top_act.index], fontsize=7)
ax.set_title('Top 10 Activities', fontweight='bold')
for i, v in enumerate(top_act.values):
    ax.text(v + 10000, i, f'{v:,}', va='center', fontsize=7)

# Activity by day of week
ax = axes[0, 1]
act_dow = act['DAY_OF_WEEK'].value_counts().reindex(dow_order)
colors_act_dow = ['#FF6B6B' if d in ['Sat', 'Sun'] else '#4ECDC4' for d in dow_order]
ax.bar(act_dow.index, act_dow.values, color=colors_act_dow, edgecolor='white')
ax.set_title('Activity by Day of Week', fontweight='bold')
for i, v in enumerate(act_dow.values):
    ax.text(i, v + 20000, f'{v/1e6:.1f}M', ha='center', fontsize=8)

# Activity by hour
ax = axes[0, 2]
act_hourly = act.groupby('ACTIVITY_HOUR').size()
colors_act_hour = ['#E71D36' if h >= 22 or h <= 5 else '#4ECDC4' for h in act_hourly.index]
ax.bar(act_hourly.index, act_hourly.values, color=colors_act_hour, edgecolor='white', width=0.8)
ax.set_title('Activity by Hour', fontweight='bold')
ax.set_xlabel('Hour')

# Login vs other split
ax = axes[1, 0]
login_activities = ['LOGIN', 'LOGOUT', 'LOGIN_FINGER', 'LOGIN_FACEID']
act['is_login'] = act['ACTIVITY_NAME'].isin(login_activities).map({True: 'Auth/Login', False: 'Other Activity'})
login_split = act['is_login'].value_counts()
ax.pie(login_split.values, labels=login_split.index, autopct='%1.1f%%',
       colors=['#FF6B6B', '#4ECDC4'], textprops={'fontsize': 11})
ax.set_title('Login vs Other Activities', fontweight='bold')

# Activity count per customer distribution
ax = axes[1, 1]
act_per_cus = act.groupby('CUSTOMER_NUMBER').size()
ax.hist(np.log10(act_per_cus), bins=50, color='#4ECDC4', edgecolor='white', alpha=0.8)
ax.set_title('Activities Per Customer (Log10)', fontweight='bold')
ax.set_xlabel('log10(Activity Count)')
ax.axvline(np.log10(act_per_cus.median()), color='red', linestyle='--',
           label=f'Median: {act_per_cus.median():.0f}')
ax.legend(fontsize=9)

# Monthly activity trend
ax = axes[1, 2]
act['month'] = act['ACTIVITY_DATE'].dt.to_period('M')
monthly_act = act.groupby('month').size()
ax.plot(range(len(monthly_act)), monthly_act.values, marker='o', color='#FF6B6B', linewidth=2)
ax.set_xticks(range(len(monthly_act)))
ax.set_xticklabels([str(m) for m in monthly_act.index], rotation=45, fontsize=8)
ax.set_title('Monthly Activity Trend', fontweight='bold')
ax.set_ylabel('Activity Count')

plt.suptitle('4.3 Activity Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fig4_activity_analysis.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig4_activity_analysis.png")



  -> Saved eda_fig4_activity_analysis.png


## 6. UNIVARIATE - FINANCIAL PRODUCTS


In [7]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# CA Balance distribution
ax = axes[0, 0]
ca_bal = dep[dep['AVG_CA_BALANCE'] > 0]['AVG_CA_BALANCE']
ax.hist(np.log10(ca_bal), bins=80, color='#4ECDC4', edgecolor='white', alpha=0.8)
ax.set_title('CA Balance Distribution (Log10)', fontweight='bold')
ax.set_xlabel('log10(CA Balance)')

# TD Balance distribution
ax = axes[0, 1]
td_bal = dep[dep['AVG_TD_BALANCE'] > 0]['AVG_TD_BALANCE']
ax.hist(np.log10(td_bal), bins=80, color='#FF6B6B', edgecolor='white', alpha=0.8)
ax.set_title('TD Balance Distribution (Log10)', fontweight='bold')
ax.set_xlabel('log10(TD Balance)')

# Monthly CA Balance trend
ax = axes[0, 2]
monthly_ca = dep.groupby(dep['MONTH'].dt.to_period('M'))['AVG_CA_BALANCE'].mean()
ax.plot(range(len(monthly_ca)), monthly_ca.values, marker='o', color='#4ECDC4', linewidth=2)
ax.set_xticks(range(len(monthly_ca)))
ax.set_xticklabels([str(m) for m in monthly_ca.index], rotation=45, fontsize=8)
ax.set_title('Monthly Avg CA Balance', fontweight='bold')

# Loan balance distribution
ax = axes[1, 0]
loan_bal = lend[lend['AVG_LOAN_AMOUNT'] > 0]['AVG_LOAN_AMOUNT']
ax.hist(np.log10(loan_bal), bins=80, color='#FFD93D', edgecolor='white', alpha=0.8)
ax.set_title('Loan Balance Distribution (Log10)', fontweight='bold')
ax.set_xlabel('log10(Loan Balance)')

# Monthly loan trend
ax = axes[1, 1]
monthly_loan = lend.groupby(lend['MONTH'].dt.to_period('M'))['AVG_LOAN_AMOUNT'].mean()
ax.plot(range(len(monthly_loan)), monthly_loan.values, marker='o', color='#FFD93D', linewidth=2)
ax.set_xticks(range(len(monthly_loan)))
ax.set_xticklabels([str(m) for m in monthly_loan.index], rotation=45, fontsize=8)
ax.set_title('Monthly Avg Loan Balance', fontweight='bold')

# Card usage
ax = axes[1, 2]
card_credit = (card['COUNT_CREDITCARD'] > 0).sum()
card_debit = (card['COUNT_DEBITCARD'] > 0).sum()
card_none = len(card) - card_credit - card_debit
card_data = [card_credit, card_debit]
card_labels = ['Credit Card', 'Debit Card (Only)']
ax.pie(card_data, labels=card_labels, autopct='%1.1f%%',
       colors=['#FF6B6B', '#4ECDC4'], textprops={'fontsize': 11})
ax.set_title('Card Product Mix', fontweight='bold')

plt.suptitle('4.4 Financial Product Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fig5_financial_products.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig5_financial_products.png")



  -> Saved eda_fig5_financial_products.png


## 7. BIVARIATE / MULTIVARIATE ANALYSIS


In [8]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Transaction vs Activity by hour
ax = axes[0, 0]
trans_hour_pct = trans.groupby('TRANS_HOUR').size() / len(trans) * 100
act_hour_pct = act.groupby('ACTIVITY_HOUR').size() / len(act) * 100
ax.plot(trans_hour_pct.index, trans_hour_pct.values, label='Transaction %',
        color='#FF6B6B', linewidth=2, marker='s', markersize=4)
ax.plot(act_hour_pct.index, act_hour_pct.values, label='Activity %',
        color='#4ECDC4', linewidth=2, marker='o', markersize=4)
ax.set_title('Hourly Pattern: Transaction vs Activity', fontweight='bold')
ax.set_xlabel('Hour')
ax.set_ylabel('% of Total')
ax.legend()
ax.axvspan(-0.5, 5.5, alpha=0.1, color='red')
ax.axvspan(21.5, 23.5, alpha=0.1, color='red')

# Day of week comparison
ax = axes[0, 1]
trans_dow_pct = trans.groupby('DAY_OF_WEEK').size().reindex(dow_order) / len(trans) * 100
act_dow_pct = act.groupby('DAY_OF_WEEK').size().reindex(dow_order) / len(act) * 100
x = np.arange(len(dow_order))
w = 0.35
ax.bar(x - w/2, trans_dow_pct.values, w, label='Transaction %', color='#FF6B6B', edgecolor='white')
ax.bar(x + w/2, act_dow_pct.values, w, label='Activity %', color='#4ECDC4', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(dow_order)
ax.set_title('Day of Week: Transaction vs Activity', fontweight='bold')
ax.set_ylabel('% of Total')
ax.legend()

# Amount by day of week
ax = axes[1, 0]
amount_dow = trans.groupby('DAY_OF_WEEK')['TRANS_AMOUNT'].agg(['mean', 'median', 'count']) \
                  .reindex(dow_order)
ax.bar(dow_order, amount_dow['mean'] / 1e6, color='#4ECDC4', edgecolor='white', alpha=0.8, label='Mean')
ax.scatter(dow_order, amount_dow['median'] / 1e6, color='red', s=100, zorder=5, label='Median')
ax.set_title('Transaction Amount by Day of Week', fontweight='bold')
ax.set_ylabel('Amount (Million VND)')
ax.legend(fontsize=9)

# Night-time transaction percentage by customer type
ax = axes[1, 1]
dep_sample = dep[dep['CUSTOMER_NUMBER'].isin(trans['CUSTOMER_NUMBER'])]
dep_agg = dep_sample.groupby('CUSTOMER_NUMBER').agg(
    avg_ca=('AVG_CA_BALANCE', 'mean'),
    avg_td=('AVG_TD_BALANCE', 'mean')
).reset_index()

trans_agg = trans.groupby('CUSTOMER_NUMBER').agg(
    total_amount=('TRANS_AMOUNT', 'sum'),
    trans_count=('TRANS_NO', 'sum'),
    night_count=('TRANS_HOUR', lambda x: ((x >= 22) | (x <= 5)).sum())
).reset_index()
trans_agg['night_pct'] = trans_agg['night_count'] / trans_agg['trans_count'] * 100

merged = pd.merge(dep_agg, trans_agg, on='CUSTOMER_NUMBER', how='inner')
merged = merged[merged['avg_ca'] > 0]
ax.scatter(np.log10(merged['avg_ca']), merged['night_pct'],
           alpha=0.3, s=1, c='#FF6B6B')
ax.set_title('CA Balance vs Night Transaction %', fontweight='bold')
ax.set_xlabel('log10(Avg CA Balance)')
ax.set_ylabel('Night Transaction %')
z = np.polyfit(np.log10(merged['avg_ca']), merged['night_pct'], 1)
p = np.poly1d(z)
x_line = np.linspace(merged['avg_ca'].apply(np.log10).min(), merged['avg_ca'].apply(np.log10).max(), 100)
ax.plot(x_line, p(x_line), color='blue', linewidth=2, linestyle='--', alpha=0.8)

plt.suptitle('5. Bivariate & Multivariate Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fig6_bivariate_analysis.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig6_bivariate_analysis.png")



  -> Saved eda_fig6_bivariate_analysis.png


## 8. CORRELATION HEATMAP


In [9]:
fig, ax = plt.subplots(figsize=(12, 10))

# Build customer-level aggregated features
cus_agg = pd.DataFrame({'CUSTOMER_NUMBER': list(cus['CUSTOMER_NUMBER'].unique())})

# Transaction features
trans_feat = trans.groupby('CUSTOMER_NUMBER').agg(
    total_trans_amount=('TRANS_AMOUNT', 'sum'),
    avg_trans_amount=('TRANS_AMOUNT', 'mean'),
    median_trans_amount=('TRANS_AMOUNT', 'median'),
    std_trans_amount=('TRANS_AMOUNT', 'std'),
    trans_count=('TRANS_NO', 'sum'),
    night_trans_ratio=('TRANS_HOUR', lambda x: ((x >= 22) | (x <= 5)).mean()),
    weekend_trans_ratio=('DAY_OF_WEEK', lambda x: x.isin(['Sat', 'Sun']).mean()),
    outside_bank_ratio=('TRANS_LV2', lambda x: (x == 'Outside_bank').mean()),
    unique_categories=('TRANS_LV2', 'nunique'),
).reset_index()

cus_agg = pd.merge(cus_agg, trans_feat, on='CUSTOMER_NUMBER', how='left')

# Activity features
act_feat = act.groupby('CUSTOMER_NUMBER').agg(
    total_activities=('ACTIVITY_NO', 'sum'),
    unique_activities=('ACTIVITY_NAME', 'nunique'),
    avg_activity_hour=('ACTIVITY_HOUR', 'mean'),
    login_ratio=('ACTIVITY_NAME', lambda x: x.isin(['LOGIN', 'LOGOUT', 'LOGIN_FINGER', 'LOGIN_FACEID']).mean()),
).reset_index()

cus_agg = pd.merge(cus_agg, act_feat, on='CUSTOMER_NUMBER', how='left')

# Deposit features
dep_feat = dep.groupby('CUSTOMER_NUMBER').agg(
    avg_ca_balance=('AVG_CA_BALANCE', 'mean'),
    avg_td_balance=('AVG_TD_BALANCE', 'mean'),
    ca_balance_std=('AVG_CA_BALANCE', 'std'),
    td_balance_std=('AVG_TD_BALANCE', 'std'),
    has_td=('COUNT_TD_ACCT', lambda x: (x > 0).any().astype(int)),
).reset_index()

cus_agg = pd.merge(cus_agg, dep_feat, on='CUSTOMER_NUMBER', how='left')

# Lending features
lend_feat = lend.groupby('CUSTOMER_NUMBER').agg(
    avg_loan_amount=('AVG_LOAN_AMOUNT', 'mean'),
    loan_count=('COUNT_OF_LOAN', 'sum'),
).reset_index()

cus_agg = pd.merge(cus_agg, lend_feat, on='CUSTOMER_NUMBER', how='left')

# Card features
card_feat = card.groupby('CUSTOMER_NUMBER').agg(
    max_credit_card=('COUNT_CREDITCARD', 'max'),
    max_debit_card=('COUNT_DEBITCARD', 'max'),
).reset_index()
card_feat['has_credit_card'] = (card_feat['max_credit_card'] > 0).astype(int)
card_feat['has_debit_card'] = (card_feat['max_debit_card'] > 0).astype(int)

cus_agg = pd.merge(cus_agg, card_feat, on='CUSTOMER_NUMBER', how='left')

# Correlation matrix
corr_cols = [c for c in cus_agg.columns if c != 'CUSTOMER_NUMBER']
corr = cus_agg[corr_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            xticklabels=True, yticklabels=True, ax=ax)
ax.set_title('Customer-Level Feature Correlation Matrix', fontweight='bold', fontsize=14)
ax.tick_params(axis='both', labelsize=7)

plt.tight_layout()
plt.savefig('eda_fig7_correlation_heatmap.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig7_correlation_heatmap.png")



  -> Saved eda_fig7_correlation_heatmap.png


## 9. FRAUD TRACK SPECIFIC: Anomaly Candidates


In [10]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Outside bank transfer outliers
ax = axes[0, 0]
outside = trans[trans['TRANS_LV2'] == 'Outside_bank']
outside_agg = outside.groupby('CUSTOMER_NUMBER').agg(
    count=('TRANS_AMOUNT', 'count'),
    total=('TRANS_AMOUNT', 'sum'),
    avg=('TRANS_AMOUNT', 'mean'),
    night_pct=('TRANS_HOUR', lambda x: ((x >= 22) | (x <= 5)).mean() * 100)
).reset_index()

# Anomaly score: total amount * night%
outside_agg['anomaly_score'] = np.log10(outside_agg['total'] + 1) * (outside_agg['night_pct'] + 1)
scatter = ax.scatter(np.log10(outside_agg['total'] + 1), outside_agg['night_pct'],
                     c=outside_agg['anomaly_score'], cmap='Reds', alpha=0.5, s=10)
ax.set_xlabel('log10(Total Outside Transfer)')
ax.set_ylabel('Night Transfer %')
ax.set_title('Fraud Anomaly Heatmap: Outside Bank Transfers', fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Anomaly Score')

# Top anomaly customers
ax = axes[0, 1]
top_anomaly = outside_agg.nlargest(15, 'anomaly_score')
bars = ax.barh(range(15), top_anomaly['anomaly_score'].values,
               color=sns.color_palette("Reds_r", 15))
ax.set_yticks(range(15))
ax.set_yticklabels([f"Cus {x}" for x in top_anomaly['CUSTOMER_NUMBER'].values], fontsize=8)
ax.set_title('Top 15 Anomaly Candidates', fontweight='bold')
ax.set_xlabel('Anomaly Score (Total Amount x Night Activity)')

# Weekly transaction pattern anomaly
ax = axes[1, 0]
# Get hourly patterns per customer
hourly_pattern = trans.groupby(['CUSTOMER_NUMBER', 'TRANS_HOUR']).size().unstack(fill_value=0)
# Normalize
hourly_pattern_norm = hourly_pattern.div(hourly_pattern.sum(axis=1), axis=0)
# Global pattern
global_pattern = trans.groupby('TRANS_HOUR').size()
global_pattern_norm = global_pattern / global_pattern.sum()

# Find most deviant customer
deviations = ((hourly_pattern_norm - global_pattern_norm) ** 2).mean(axis=1).sort_values(ascending=False)
top_deviant = deviations.index[:5]

colors_deviant = ['#FF6B6B', '#4ECDC4', '#FFD93D', '#95E1D3', '#F38181']
for i, cus_id in enumerate(top_deviant):
    ax.plot(hourly_pattern_norm.columns, hourly_pattern_norm.loc[cus_id].values * 100,
            label=f'Cus {cus_id}', color=colors_deviant[i], linewidth=1.5, alpha=0.8)
ax.plot(global_pattern_norm.index, global_pattern_norm.values * 100,
        label='Global', color='black', linewidth=3, linestyle='--')
ax.set_title('Most Deviant Transaction Patterns', fontweight='bold')
ax.set_xlabel('Hour')
ax.set_ylabel('% of Transactions')
ax.legend(fontsize=7)

# Unusual transaction days
ax = axes[1, 1]
weekend_trans = trans[trans['DAY_OF_WEEK'].isin(['Sat', 'Sun'])]
weekend_large = weekend_trans[weekend_trans['TRANS_AMOUNT'] > trans['TRANS_AMOUNT'].quantile(0.95)]
weekend_large_count = weekend_large.groupby('CUSTOMER_NUMBER').size().sort_values(ascending=False)
ax.hist(weekend_large_count.values, bins=50, color='#E71D36', edgecolor='white', alpha=0.8)
ax.set_title('Large Weekend Transactions per Customer', fontweight='bold')
ax.set_xlabel('Count of Large Weekend Transactions (>P95)')
ax.set_ylabel('Customer Count')
ax.axvline(x=5, color='red', linestyle='--', label='Threshold=5')
ax.legend()

plt.suptitle('6. Track A: Fraud & Anomaly Detection', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fig8_fraud_track.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig8_fraud_track.png")



  -> Saved eda_fig8_fraud_track.png


## 10. NBFO TRACK SPECIFIC: Product Adoption Analysis


In [11]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Product adoption timeline
ax = axes[0, 0]
monthly_cus_count = dep.groupby(dep['MONTH'].dt.to_period('M'))['CUSTOMER_NUMBER'].nunique()
monthly_lend_count = lend.groupby(lend['MONTH'].dt.to_period('M'))['CUSTOMER_NUMBER'].nunique()
monthly_card_count = card.groupby(card['MONTH'].dt.to_period('M'))['CUSTOMER_NUMBER'].nunique()

months = sorted(set(list(monthly_cus_count.index) + list(monthly_lend_count.index) + list(monthly_card_count.index)))
x = range(len(months))
ax.plot(x, [monthly_cus_count.get(m, 0) for m in months], marker='o', label='Deposit Users', color='#4ECDC4')
ax.plot(x, [monthly_lend_count.get(m, 0) for m in months], marker='s', label='Lending Users', color='#FF6B6B')
ax.plot(x, [monthly_card_count.get(m, 0) for m in months], marker='^', label='Card Users', color='#FFD93D')
ax.set_xticks(x)
ax.set_xticklabels([str(m) for m in months], rotation=45, fontsize=8)
ax.set_title('Product Adoption Over Time', fontweight='bold')
ax.set_ylabel('Active Customers')
ax.legend()

# Balance growth before product adoption
ax = axes[0, 1]
# Customers who have lending
lend_cus = set(lend['CUSTOMER_NUMBER'].unique())
dep_with_lend = dep[dep['CUSTOMER_NUMBER'].isin(lend_cus)]
dep_no_lend = dep[~dep['CUSTOMER_NUMBER'].isin(lend_cus)]

monthly_ca_lend = dep_with_lend.groupby(dep_with_lend['MONTH'].dt.to_period('M'))['AVG_CA_BALANCE'].mean()
monthly_ca_nolend = dep_no_lend.groupby(dep_no_lend['MONTH'].dt.to_period('M'))['AVG_CA_BALANCE'].mean()

common_months = sorted(set(monthly_ca_lend.index) & set(monthly_ca_nolend.index))
ax.plot(range(len(common_months)), [monthly_ca_lend[m] for m in common_months],
        marker='o', label='With Loan', color='#FF6B6B')
ax.plot(range(len(common_months)), [monthly_ca_nolend[m] for m in common_months],
        marker='s', label='Without Loan', color='#4ECDC4')
ax.set_xticks(range(len(common_months)))
ax.set_xticklabels([str(m) for m in common_months], rotation=45, fontsize=8)
ax.set_title('Avg CA Balance: Loan vs No Loan', fontweight='bold')
ax.set_ylabel('Avg CA Balance')
ax.legend()

# Propensity indicators: rising balance before new product
ax = axes[1, 0]
# Customers who first got a card in month 3+ (excluding month 1-2)
card_first = card.groupby('CUSTOMER_NUMBER')['MONTH'].min().reset_index()
card_first.columns = ['CUSTOMER_NUMBER', 'first_card_month']
card_first = card_first[card_first['first_card_month'] > card['MONTH'].min()]

# Get their deposit trend before first card
card_adopters = card_first['CUSTOMER_NUMBER'].unique()[:500]  # Sample for performance
balances_before = []
for cus_id in card_adopters:
    cus_dep = dep[dep['CUSTOMER_NUMBER'] == cus_id].sort_values('MONTH')
    if len(cus_dep) >= 3:
        balances_before.append(cus_dep['AVG_CA_BALANCE'].values[:3])

if balances_before:
    balances_array = np.array([b for b in balances_before if len(b) == 3])
    if len(balances_array) > 0:
        avg_trend = balances_array.mean(axis=0)
        ax.plot(['M-2', 'M-1', 'M0'], avg_trend / 1e6, marker='o', color='#FF6B6B', linewidth=2)
        ax.fill_between(['M-2', 'M-1', 'M0'],
                         (avg_trend - balances_array.std(axis=0)) / 1e6,
                         (avg_trend + balances_array.std(axis=0)) / 1e6,
                         alpha=0.2, color='#FF6B6B')
ax.set_title('CA Balance Trend Before Getting First Card', fontweight='bold')
ax.set_ylabel('Avg CA Balance (Million VND)')
ax.set_xlabel('Months Relative to First Card')

# Cross-sell matrix
ax = axes[1, 1]
cus_products = pd.DataFrame({'CUSTOMER_NUMBER': list(cus['CUSTOMER_NUMBER'].unique())})
cus_products['has_deposit'] = cus_products['CUSTOMER_NUMBER'].isin(dep_ids).astype(int)
cus_products['has_lending'] = cus_products['CUSTOMER_NUMBER'].isin(lend_ids).astype(int)
cus_products['has_card'] = cus_products['CUSTOMER_NUMBER'].isin(card_ids).astype(int)

cross_data = pd.crosstab(cus_products['has_deposit'], cus_products['has_lending'])
# Simplified visualization
products = ['Deposit', 'Lending', 'Card']
product_sets = [dep_ids, lend_ids, card_ids]
overlap = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        overlap[i, j] = len(product_sets[i] & product_sets[j])

sns.heatmap(overlap, annot=True, fmt=',.0f', cmap='YlOrRd',
            xticklabels=products, yticklabels=products, ax=ax)
ax.set_title('Product Overlap Matrix (# Customers)', fontweight='bold')

plt.suptitle('6. Track B: Next Best Financial Offer (NBFO)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fig9_nbfo_track.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig9_nbfo_track.png")



  -> Saved eda_fig9_nbfo_track.png


## 11. SEGMENTATION TRACK SPECIFIC


In [12]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Behavioral diversity
ax = axes[0, 0]
cus_diversity = act.groupby('CUSTOMER_NUMBER')['ACTIVITY_NAME'].nunique()
ax.hist(cus_diversity, bins=30, color='#4ECDC4', edgecolor='white')
ax.set_title('Activity Diversity per Customer', fontweight='bold')
ax.set_xlabel('Unique Activity Types')
ax.set_ylabel('Customer Count')
ax.axvline(cus_diversity.median(), color='red', linestyle='--', label=f'Median: {cus_diversity.median():.0f}')
ax.legend()

# Digital engagement score
ax = axes[0, 1]
eng_features = cus_agg[['total_activities', 'unique_activities', 'trans_count',
                          'unique_categories', 'avg_ca_balance', 'has_credit_card']].dropna()
# Simple engagement scoring (normalize and sum)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
eng_scaled = pd.DataFrame(scaler.fit_transform(eng_features.fillna(0)), columns=eng_features.columns)
eng_features['engagement_score'] = eng_scaled.mean(axis=1)
ax.hist(eng_features['engagement_score'], bins=50, color='#FF6B6B', edgecolor='white')
ax.set_title('Digital Engagement Score Distribution', fontweight='bold')
ax.set_xlabel('Engagement Score (Z-score avg)')

# Cluster tendency preview
ax = axes[1, 0]
# Use two key features for visualization
viz_data = cus_agg[['trans_count', 'total_activities', 'avg_ca_balance']].dropna()
viz_data = viz_data[(viz_data['trans_count'] > 0) & (viz_data['total_activities'] > 0)]
sample_viz = viz_data.sample(min(2000, len(viz_data)), random_state=42)
sc = ax.scatter(np.log10(sample_viz['total_activities']),
                np.log10(sample_viz['avg_ca_balance'] + 1),
                c=np.log10(sample_viz['trans_count'] + 1),
                cmap='viridis', alpha=0.6, s=15)
ax.set_xlabel('log10(Total Activities)')
ax.set_ylabel('log10(Avg CA Balance)')
ax.set_title('Customer Segmentation Preview\nColor = Transaction Count', fontweight='bold')
plt.colorbar(sc, ax=ax, label='log10(Trans Count)')

# Customer archetypes summary
ax = axes[1, 1]
# Simple clustering for archetypes
from sklearn.cluster import KMeans
cluster_data = cus_agg[['trans_count', 'total_activities', 'avg_ca_balance',
                         'unique_categories', 'night_trans_ratio']].fillna(0)
# Filter to active customers
cluster_data = cluster_data[(cluster_data['trans_count'] > 0) | (cluster_data['total_activities'] > 0)]
cluster_sample = cluster_data.sample(min(5000, len(cluster_data)), random_state=42)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_sample['cluster'] = kmeans.fit_predict(
    StandardScaler().fit_transform(cluster_sample))

cluster_summary = cluster_sample.groupby('cluster').agg(
    size=('trans_count', 'count'),
    avg_trans=('trans_count', 'mean'),
    avg_activity=('total_activities', 'mean'),
    avg_balance=('avg_ca_balance', 'mean'),
    avg_night=('night_trans_ratio', 'mean'),
).round(1)

# Plot cluster profiles
cluster_summary_norm = cluster_summary.copy()
for col in ['avg_trans', 'avg_activity', 'avg_balance', 'avg_night']:
    cluster_summary_norm[col] = cluster_summary_norm[col] / cluster_summary_norm[col].max()

archetypes = ['Passive', 'Transactor', 'Digital Native', 'Power User']
ax.barh(archetypes, cluster_summary_norm['avg_trans'].values, color='#4ECDC4',
        alpha=0.7, label='Transaction')
ax.barh(archetypes, cluster_summary_norm['avg_activity'].values, color='#FF6B6B',
        alpha=0.7, left=cluster_summary_norm['avg_trans'].values, label='Activity')
ax.set_title('Customer Archetype Profiles (Preliminary)', fontweight='bold')
ax.set_xlabel('Normalized Score')
ax.legend(fontsize=8, loc='lower right')

plt.suptitle('6. Track C: Persona-Based Segmentation', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fig10_segmentation_track.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig10_segmentation_track.png")



  -> Saved eda_fig10_segmentation_track.png


## 12. TEMPORAL HEATMAPS


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Transaction heatmap: Day of Week x Hour
trans_heatmap = trans.pivot_table(index='DAY_OF_WEEK', columns='TRANS_HOUR',
                                   values='CUSTOMER_NUMBER', aggfunc='count')
trans_heatmap = trans_heatmap.reindex(dow_order)
sns.heatmap(trans_heatmap, cmap='YlOrRd', ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Transaction Density: Day x Hour', fontweight='bold')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Day of Week')

# Activity heatmap: Day of Week x Hour
act_heatmap = act.pivot_table(index='DAY_OF_WEEK', columns='ACTIVITY_HOUR',
                               values='CUSTOMER_NUMBER', aggfunc='count')
act_heatmap = act_heatmap.reindex(dow_order)
sns.heatmap(act_heatmap, cmap='YlGnBu', ax=axes[1], cbar_kws={'label': 'Count'})
axes[1].set_title('Activity Density: Day x Hour', fontweight='bold')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Day of Week')

plt.suptitle('5.3 Temporal Analysis - Heatmaps', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_fig11_temporal_heatmaps.png', bbox_inches='tight')
plt.close()
print("  -> Saved eda_fig11_temporal_heatmaps.png")

print("\n" + "="*60)
print("EDA VISUALIZATIONS COMPLETE!")
print("="*60)
print("Generated 11 figures:")
print("  1. eda_fig1_missing_values.png")
print("  2. eda_fig2_customer_demographics.png")
print("  3. eda_fig3_transaction_analysis.png")
print("  4. eda_fig4_activity_analysis.png")
print("  5. eda_fig5_financial_products.png")
print("  6. eda_fig6_bivariate_analysis.png")
print("  7. eda_fig7_correlation_heatmap.png")
print("  8. eda_fig8_fraud_track.png")
print("  9. eda_fig9_nbfo_track.png")
print("  10. eda_fig10_segmentation_track.png")
print("  11. eda_fig11_temporal_heatmaps.png")


  -> Saved eda_fig11_temporal_heatmaps.png

EDA VISUALIZATIONS COMPLETE!
Generated 11 figures:
  1. eda_fig1_missing_values.png
  2. eda_fig2_customer_demographics.png
  3. eda_fig3_transaction_analysis.png
  4. eda_fig4_activity_analysis.png
  5. eda_fig5_financial_products.png
  6. eda_fig6_bivariate_analysis.png
  7. eda_fig7_correlation_heatmap.png
  8. eda_fig8_fraud_track.png
  9. eda_fig9_nbfo_track.png
  10. eda_fig10_segmentation_track.png
  11. eda_fig11_temporal_heatmaps.png
